In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("../../")
coltekin = pd.read_csv(PROJECT_ROOT / "data" / "coltekin.csv")
results = pd.read_csv(PROJECT_ROOT / "results" / "3_analyze" / "total_results.csv")

METRIC_MAP = {
    "TTR": "type_token_ratio",
    "IiS": "compress_reduction_pct",
    "H": "token_frequency_entropy",
}


def process_column(coltekin_df, col_name, results_df, metric_col):
    """Process one coltekin column into a ranked comparison dataframe."""
    s = coltekin_df[col_name].dropna().reset_index(drop=True)
    df = pd.DataFrame({"lang": s.values, "original_rank": np.arange(1, len(s) + 1)})

    agg = df.groupby("lang").agg(
        coltekin_rank=("original_rank", "mean"),
        best_position=("original_rank", "min"),
    ).reset_index()

    agg = agg.sort_values(["coltekin_rank", "best_position"]).reset_index(drop=True)
    agg["coltekin_rank"] = np.arange(1, len(agg) + 1)
    agg = agg.drop(columns="best_position")

    merged = agg.merge(results_df[["lang", "language", metric_col]].dropna(subset=[metric_col]), on="lang", how="inner")

    merged["my_rank"] = merged[metric_col].rank(ascending=False, method="min").astype(int)

    return merged[["lang", "language", "coltekin_rank", metric_col, "my_rank"]]


dfs = {}
for col, metric in METRIC_MAP.items():
    dfs[col] = process_column(coltekin, col, results, metric)
    print(f"\n=== {col} (metric: {metric}) ===")
    print(dfs[col].head(10).to_string(index=False))


=== TTR (metric: type_token_ratio) ===
lang  language  coltekin_rank  type_token_ratio  my_rank
 kor    Korean              1          2.370185        1
 fin   Finnish              2          0.747095        2
 est  Estonian              3          0.206968        6
 rus   Russian              4          0.142080        9
 ukr Ukrainian              5          0.139753       10
 lav   Latvian              6          0.061093       24
 pol    Polish              7          0.126369       13
 slk    Slovak              8          0.116860       15
 ces     Czech              9          0.127477       12
 slv Slovenian             10          0.086468       18

=== IiS (metric: compress_reduction_pct) ===
lang           language  coltekin_rank  compress_reduction_pct  my_rank
 hun          Hungarian              1                   26.29       10
 tur            Turkish              2                   27.08        5
 eus             Basque              3                   25.65       15

In [2]:
from scipy.stats import spearmanr, kendalltau

for col, metric in METRIC_MAP.items():
    df = dfs[col]
    rho, p_rho = spearmanr(df["coltekin_rank"], df["my_rank"])
    tau, p_tau = kendalltau(df["coltekin_rank"], df["my_rank"])
    print(f"{col} ({metric}): n={len(df)}  Spearman ρ={rho:.3f} (p={p_rho:.2e})  Kendall τ={tau:.3f} (p={p_tau:.2e})")

TTR (type_token_ratio): n=37  Spearman ρ=0.630 (p=2.92e-05)  Kendall τ=0.483 (p=2.54e-05)
IiS (compress_reduction_pct): n=40  Spearman ρ=0.313 (p=4.90e-02)  Kendall τ=0.236 (p=3.20e-02)
H (token_frequency_entropy): n=40  Spearman ρ=0.743 (p=3.92e-08)  Kendall τ=0.613 (p=2.56e-08)


In [ ]:
for col, metric in METRIC_MAP.items():
    df = dfs[col].copy()
    df["rank_diff"] = (df["my_rank"] - df["coltekin_rank"]).abs()
    worst = df.nlargest(5, "rank_diff")[["lang", "language_name", "coltekin_rank", "my_rank", "rank_diff"]]
    print(f"\n=== {col} ({metric}) — biggest rank differences ===")
    print(worst.to_string(index=False))


=== TTR (type_token_ratio) — biggest rank differences ===
lang  language  coltekin_rank  my_rank  rank_diff
 uig    Uyghur             36        4         32
 lat     Latin             31        5         26
 lav   Latvian              6       24         18
 hun Hungarian             17        3         14
 spa   Spanish             20       34         14

=== IiS (compress_reduction_pct) — biggest rank differences ===
lang         language  coltekin_rank  my_rank  rank_diff
 cmn Mandarin Chinese              4       40         36
 ind       Indonesian             38        3         35
 vie       Vietnamese             37        9         28
 afr        Afrikaans              5       31         26
 lat            Latin              7       33         26

=== H (token_frequency_entropy) — biggest rank differences ===
lang   language  coltekin_rank  my_rank  rank_diff
 heb     Hebrew             40        6         34
 lat      Latin             31        7         24
 uig     Uyghur  